# BeyondSmile: A Challenge on Detecting Depression through Facial Behavior and Head Gestures


In [12]:
import pandas as pd
import numpy as np
import tsfel
import neurokit2 as nk
import matplotlib.pyplot as plt
import datetime
import json
import pycatch22


## Load and Parse the Data
The dataset will likely be provided as JSON files, each containing metadata for a single sample or for multiple samples. Below is an example of a single data entry.

We will parse this example and show how to access different features.

In [13]:
# Read the columns for the data

In [14]:
columns_tself = pd.read_csv('./columns_mid_tsfel_AU.csv')

In [15]:
del columns_tself['Unnamed: 0']

In [16]:
tself_columns_mid = list(columns_tself.columns)


In [17]:
columns_pycatch = pd.read_csv('./columns_mid_pycatch_AU.csv')

In [18]:
del columns_pycatch['Unnamed: 0']

In [19]:
pycatch_columns_mid = list(columns_pycatch.columns)


In [20]:
tself_columns_mor = [name.replace('mid', 'mor') for name in tself_columns_mid]
tself_columns_aft = [name.replace('mid', 'aft') for name in tself_columns_mid]
tself_columns_eve = [name.replace('mid', 'eve') for name in tself_columns_mid]

In [21]:
pycatch_columns_mor = [name.replace('mid', 'mor') for name in pycatch_columns_mid]
pycatch_columns_aft = [name.replace('mid', 'aft') for name in pycatch_columns_mid]
pycatch_columns_eve = [name.replace('mid', 'eve') for name in pycatch_columns_mid]

In [22]:
data_phq = pd.read_csv('./dataset/groundtruth/phq9 _date.csv')
data_phq

,pid,start_ts,end_ts,start_phq9,end_phq9,depression_episode
0,P08,7/21/22,08/09/2022,6,1.0,0
1,P08,08/09/2022,8/23/22,1,9.0,0
2,P10,7/21/22,08/09/2022,8,7.0,1
3,P10,08/09/2022,09/02/2022,7,2.0,0
4,P12,7/22/22,08/09/2022,10,12.0,1
5,P12,08/09/2022,8/23/22,12,9.0,1
6,P13,7/25/22,08/09/2022,1,3.0,0
7,P13,08/09/2022,8/23/22,3,2.0,0
8,P14,7/25/22,08/08/2022,11,NaN,0
9,P15,7/26/22,08/10/2022,4,9.0,0


# Read the labels

In [23]:
def closest_index(target_value, timeline_list):
    differences = np.abs(np.array(timeline_list) - target_value)
    closest_index = differences.argmin()

    return closest_index
    

In [24]:
for record in range(0, len(data_phq)):
    data_patient_depression = data_phq.loc[record]
    patient = data_phq.loc[record]['pid']
    diagnosis = data_phq.loc[record]['depression_episode']
    start_monitoring = data_patient_depression.start_ts
    end_monitoring = data_patient_depression.end_ts
    #Change the dates into the timestamp
    element_start = datetime.datetime(2022, int(start_monitoring.split('/')[0]), int(start_monitoring.split('/')[1]))
    timestamp_start = datetime.datetime.timestamp(element_start)
    element_end = datetime.datetime(2022, int(end_monitoring.split('/')[0]), int(end_monitoring.split('/')[1]))
    timestamp_end = datetime.datetime.timestamp(element_end)
    #Reorder the data
    with open('./dataset/data/'+patient+ '.json', 'r') as f:
            data = json.load(f)

    times = []
    numbers = []
    for i in range(0, len(data)):
        times.append(int(data[i]['timestamp'])/1000)
        numbers.append(i)
    min_value = datetime.datetime.fromtimestamp(min(times)).isoformat()
    max_value = datetime.datetime.fromtimestamp(max(times)).isoformat()

    b = enumerate(times)
    c = sorted(b, key = lambda i:i[1])
    times_index_primary = []

    for e in c:
        times_index_primary.append(e[0])

    sorted_times = sorted(times)

    # Reorder data json
    data2 = []
    for i in range(0, len(times_index_primary)):
        data2.append(data[times_index_primary[i]])

    times = []
    numbers = []
    for i in range(0, len(data2)):
        times.append(int(data2[i]['timestamp'])/1000)
        numbers.append(i)

    # Find the beginning of the record and end of the record
    counter_start = 0
    while(sorted_times[counter_start]<timestamp_start):
        counter_start +=1

    counter_end = counter_start
    for i in range(counter_start, len(sorted_times)):
        if sorted_times[counter_end]<=timestamp_end:
            counter_end +=1
        else:
            break
    counter_end = counter_end - 1

    #Select subdataset
    data3 = data2[counter_start:counter_end+1]

    timeline = []
    timelinedate = []
    for time in range(0, len(data3)):
        timeline.append(float(data3[time]['timestamp'])/1000)
        timelinedate.append(datetime.datetime.fromtimestamp(float(data3[time]['timestamp'])/1000))

    # Define separete subdata for the midningt, morning, afternoon and evening 
    start_index_list = []
    end_index_list = []

    start_index = 0

    for portion in range(0, 100):

        flag =0
        start_value = timeline[start_index]


        target_value = start_value +60*60*24 #1 day more

        end_index = closest_index(target_value, timeline) 
        end_value = timeline[end_index]




        if (end_value - start_value) > 60*60*24:
            flag=1
            start_index_list.append(start_index)
            end_index_list.append(end_index - 1)

          #  print(datetime.datetime.fromtimestamp(timeline[start_index]))
          #  print(datetime.datetime.fromtimestamp(timeline[end_index-1]))

            start_index = end_index
        else:

            if (end_index+1)<len(timeline):
                if (timeline[end_index+1] - start_value) > 60*60*24:
                    flag =1
                    start_index_list.append(start_index)
                    end_index_list.append(end_index)

                 #   print(datetime.datetime.fromtimestamp(timeline[start_index]))
                 #   print(datetime.datetime.fromtimestamp(timeline[end_index-1]))


                    start_index = end_index +1




        if flag ==0:
            break

    sub_data = pd.DataFrame(columns=['record', 'start_subrecord', 'end_subrecord'])

    sub_data['start_subrecord'] = start_index_list
    sub_data['end_subrecord'] = end_index_list
    sub_data['record'] = 0
    sub_data['diagnosis'] = diagnosis

    if (timeline[-1] - timeline[0])<=86400:
        sub_data['start_subrecord'] = [0]
        sub_data['end_subrecord']= [len(data3)]
        sub_data['record']= [0]
        sub_data['diagnosis']= [diagnosis]

    if len(sub_data)>0:
        sample = 0
    else:
        sample = -1

    if sample==0:
        start_index = sub_data.loc[sample]['start_subrecord']
        end_index = sub_data.loc[sample]['end_subrecord']
        data_test = data3[start_index:end_index+1]
        # Select the day for 4 periods: midnight (12am-6am), morning (6am-12pm), afternoon (12pm-6pm), and evening (6pm12am) (to daytime!)

        midnight_time = []
        morning_time = []
        afternoon_time = []
        evening_time = []
        for i in range(0, len(data_test)):
            hour_sample = datetime.datetime.fromtimestamp(float(data_test[i]['timestamp'])/1000).hour
            if hour_sample>=0 and hour_sample<6:
                midnight_time.append(i)
            if hour_sample>=6 and hour_sample<12:
                morning_time.append(i)
            if hour_sample>=12 and hour_sample<18:
                afternoon_time.append(i)
            if hour_sample>=18 and hour_sample<=23:
                evening_time.append(i)

        data_midnight = []
        for i in range(0, len(midnight_time)):
            data_midnight.append(data_test[midnight_time[i]])

        data_morning = []
        for i in range(0, len(morning_time)):
            data_morning.append(data_test[morning_time[i]])

        data_afternoon = []
        for i in range(0, len(afternoon_time)):
            data_afternoon.append(data_test[afternoon_time[i]])

        data_evening = []
        for i in range(0, len(evening_time)):
            data_evening.append(data_test[evening_time[i]])





        # Define separete subdata for the midningt, morning, afternoon and evening 

        COLUMN_NAMES = ['AU01', 'AU02', 'AU04', 'AU06', 'AU07', 'AU10', 'AU12', 'AU14', 'AU15', 'AU17', 'AU23', 'AU24']
        COLUMN_NAMES_mid = []
        for i in range(0, len(COLUMN_NAMES)):
            COLUMN_NAMES_mid.append(COLUMN_NAMES[i] +'_mid')
        records_AU_mid = pd.DataFrame(columns= COLUMN_NAMES_mid)

        for i in range(0, len(data_midnight)):
            au_data = data_midnight[i]['au']
            au_names = list(au_data.keys())
            au_values = list(au_data.values())
            # Convert logit values (au_values) to probabilities
            au_probabilities = [1/(1 + np.exp(-v)) for v in au_values]

            if len(au_probabilities)!=0:
                records_AU_mid.loc[i] = au_probabilities
            else: 
                records_AU_mid.loc[i] = [np.nan]*12

        records_AU_mid_cleaned = records_AU_mid.copy()
        records_AU_mid_cleaned = records_AU_mid_cleaned.dropna()

        if len(records_AU_mid_cleaned) >=12:     # Retrieves a pre-defined feature configuration file to extract the temporal, statistical and spectral feature sets
            cfg = tsfel.get_features_by_domain()

            # Extract features
            X = tsfel.time_series_features_extractor(cfg, records_AU_mid_cleaned)
            X_mid_to_delete = []
            for name in X.columns:
                if 'Spectrogram mean coefficient_' in name:
                    X_mid_to_delete.append(name)
            X = X.drop(X_mid_to_delete, axis=1)

        else:
            X = pd.DataFrame(columns=tself_columns_mid)
            X.loc[0] = [np.nan]*1488

        data_action_mid = X.copy()

        if len(records_AU_mid_cleaned)>3:    
            for j in range(0, len(COLUMN_NAMES_mid)):
                name_action = COLUMN_NAMES_mid[j]
                features_pycatch = pycatch22.catch22_all(records_AU_mid_cleaned[name_action])
                COLUMN = []
                for i in range(0, len(features_pycatch['names'])):
                    COLUMN.append(name_action+ '_' + features_pycatch['names'][i]) 
                features_action_sub_mid = pd.DataFrame(columns=COLUMN)
                features_action_sub_mid.loc[0] = features_pycatch['values']
                if j == 0:
                    features_action_mid = features_action_sub_mid.copy()
                else:
                    features_action_mid = pd.concat([features_action_mid, features_action_sub_mid], axis=1)
        else:
            features_action_mid = pd.DataFrame(columns=pycatch_columns_mid)
            features_action_mid.loc[0] = [np.nan]*264 


        data_action_mid = pd.concat([data_action_mid, features_action_mid], axis=1)

        approx_entropy_columns = [name + '_app_ent' for name in records_AU_mid_cleaned.columns]
        data_approx_entropy_mid = pd.DataFrame(columns=approx_entropy_columns)
        app_ent_AU_mid= []

        for i in range(0, len(approx_entropy_columns)): 
            try:
                approximate_entropy, parameters = nk.entropy_approximate(records_AU_mid_cleaned[records_AU_mid_cleaned.columns[i]])
                 # Approximate entropy
            except:
                approximate_entropy = 0
            app_ent_AU_mid.append(approximate_entropy)
        data_approx_entropy_mid.loc[0] = app_ent_AU_mid

        data_action_mid = pd.concat([data_action_mid, data_approx_entropy_mid], axis=1)

        rsd_columns_mid = [name + '_rsd' for name in records_AU_mid_cleaned.columns]
        data_rsd_mid = pd.DataFrame(columns=rsd_columns_mid)
        rsd_AU_mid = []
        for i in range(0, len(rsd_columns_mid)): 
            rsd = 100*np.std(records_AU_mid_cleaned[records_AU_mid_cleaned.columns[i]])/(np.mean(records_AU_mid_cleaned[records_AU_mid_cleaned.columns[i]])+0.00000000000000000000001)
            rsd_AU_mid.append(rsd)
        data_rsd_mid.loc[0] = rsd_AU_mid

        data_action_mid = pd.concat([data_action_mid, data_rsd_mid], axis=1)

        # Extract feautres for morning
        COLUMN_NAMES = ['AU01', 'AU02', 'AU04', 'AU06', 'AU07', 'AU10', 'AU12', 'AU14', 'AU15', 'AU17', 'AU23', 'AU24']
        COLUMN_NAMES_mor = []
        for i in range(0, len(COLUMN_NAMES)):
            COLUMN_NAMES_mor.append(COLUMN_NAMES[i] +'_mor')
        records_AU_mor = pd.DataFrame(columns= COLUMN_NAMES_mor)
        for i in range(0, len(data_morning)):
            au_data = data_morning[i]['au']
            au_names = list(au_data.keys())
            au_values = list(au_data.values())
            # Convert logit values (au_values) to probabilities
            au_probabilities = [1/(1 + np.exp(-v)) for v in au_values]

            if len(au_probabilities)!=0:
                records_AU_mor.loc[i] = au_probabilities
            else: 
                records_AU_mor.loc[i] = [np.nan]*12

        records_AU_mor_cleaned = records_AU_mor.copy()
        records_AU_mor_cleaned = records_AU_mor_cleaned.dropna()

        print(records_AU_mor_cleaned)

        if len(records_AU_mor_cleaned) >=12:     # Retrieves a pre-defined feature configuration file to extract the temporal, statistical and spectral feature sets
            cfg = tsfel.get_features_by_domain()

            # Extract features
            X = tsfel.time_series_features_extractor(cfg, records_AU_mor_cleaned)
            X_mor_to_delete = []
            for name in X.columns:
                if 'Spectrogram mean coefficient_' in name:
                    X_mor_to_delete.append(name)
            X = X.drop(X_mor_to_delete, axis=1)

        else:
            X = pd.DataFrame(columns=tself_columns_mor)
            X.loc[0] = [np.nan]*1488

        data_action_mor = X.copy()

        if len(records_AU_mor_cleaned)>3:    
            for j in range(0, len(COLUMN_NAMES_mor)):
                name_action = COLUMN_NAMES_mor[j]
                features_pycatch = pycatch22.catch22_all(records_AU_mor_cleaned[name_action])
                COLUMN = []
                for i in range(0, len(features_pycatch['names'])):
                    COLUMN.append(name_action+ '_' + features_pycatch['names'][i]) 
                features_action_sub_mor = pd.DataFrame(columns=COLUMN)
                features_action_sub_mor.loc[0] = features_pycatch['values']
                if j == 0:
                    features_action_mor = features_action_sub_mor.copy()
                else:
                    features_action_mor = pd.concat([features_action_mor, features_action_sub_mor], axis=1)
        else:
            features_action_mor = pd.DataFrame(columns=pycatch_columns_mor)
            features_action_mor.loc[0] = [np.nan]*264




        data_action_mor = pd.concat([data_action_mor, features_action_mor], axis=1)

        approx_entropy_columns = [name + '_app_ent' for name in records_AU_mor_cleaned.columns]
        data_approx_entropy_mor = pd.DataFrame(columns=approx_entropy_columns)
        app_ent_AU_mor= []

        for i in range(0, len(approx_entropy_columns)): 
            try:
                approximate_entropy, parameters = nk.entropy_approximate(records_AU_mor_cleaned[records_AU_mor_cleaned.columns[i]])
                 # Approximate entropy
            except:
                approximate_entropy = 0
            app_ent_AU_mor.append(approximate_entropy)
        data_approx_entropy_mor.loc[0] = app_ent_AU_mor
        data_action_mor = pd.concat([data_action_mor, data_approx_entropy_mor], axis=1)
        rsd_columns_mor= [name + '_rsd' for name in records_AU_mor_cleaned.columns]
        data_rsd_mor = pd.DataFrame(columns=rsd_columns_mor)
        rsd_AU_mor = []
        for i in range(0, len(rsd_columns_mor)): 
            rsd = 100*np.std(records_AU_mor_cleaned[records_AU_mor_cleaned.columns[i]])/(np.mean(records_AU_mor_cleaned[records_AU_mor_cleaned.columns[i]])+0.00000000000000000000001)
            rsd_AU_mor.append(rsd)
        data_rsd_mor.loc[0] = rsd_AU_mor

        data_action_mor = pd.concat([data_action_mor, data_rsd_mor], axis=1)
        # Create records for afternoon

        COLUMN_NAMES = ['AU01', 'AU02', 'AU04', 'AU06', 'AU07', 'AU10', 'AU12', 'AU14', 'AU15', 'AU17', 'AU23', 'AU24']
        COLUMN_NAMES_aft = []
        for i in range(0, len(COLUMN_NAMES)):
            COLUMN_NAMES_aft.append(COLUMN_NAMES[i] +'_aft')
        records_AU_aft = pd.DataFrame(columns= COLUMN_NAMES_aft)

        for i in range(0, len(data_afternoon)):
            au_data = data_afternoon[i]['au']
            au_names = list(au_data.keys())
            au_values = list(au_data.values())
            # Convert logit values (au_values) to probabilities
            au_probabilities = [1/(1 + np.exp(-v)) for v in au_values]

            if len(au_probabilities)!=0:
                records_AU_aft.loc[i] = au_probabilities
            else: 
                records_AU_aft.loc[i] = [np.nan]*12

        records_AU_aft_cleaned = records_AU_aft.copy()
        records_AU_aft_cleaned = records_AU_aft_cleaned.dropna()

        if len(records_AU_aft_cleaned) >=12:     # Retrieves a pre-defined feature configuration file to extract the temporal, statistical and spectral feature sets
            cfg = tsfel.get_features_by_domain()

            # Extract features
            X = tsfel.time_series_features_extractor(cfg, records_AU_aft_cleaned)
            X_aft_to_delete = []
            for name in X.columns:
                if 'Spectrogram mean coefficient_' in name:
                    X_aft_to_delete.append(name)
            X = X.drop(X_aft_to_delete, axis=1)

        else:
            X = pd.DataFrame(columns=tself_columns_aft)
            X.loc[0] = [np.nan]*1488

        data_action_aft = X.copy()

        if len(records_AU_aft_cleaned)>3:    
            for j in range(0, len(COLUMN_NAMES_aft)):
                name_action = COLUMN_NAMES_aft[j]
                features_pycatch = pycatch22.catch22_all(records_AU_aft_cleaned[name_action])
                COLUMN = []
                for i in range(0, len(features_pycatch['names'])):
                    COLUMN.append(name_action+ '_' + features_pycatch['names'][i]) 
                features_action_sub_aft = pd.DataFrame(columns=COLUMN)
                features_action_sub_aft.loc[0] = features_pycatch['values']
                if j == 0:
                    features_action_aft = features_action_sub_aft.copy()
                else:
                    features_action_aft = pd.concat([features_action_aft, features_action_sub_aft], axis=1)
        else:
            features_action_aft = pd.DataFrame(columns=pycatch_columns_aft)
            features_action_aft.loc[0] = [np.nan]*264

        data_action_aft = pd.concat([data_action_aft, features_action_aft], axis=1)
        approx_entropy_columns = [name + '_app_ent' for name in records_AU_aft_cleaned.columns]
        data_approx_entropy_aft = pd.DataFrame(columns=approx_entropy_columns)
        app_ent_AU_aft= []


        for i in range(0, len(approx_entropy_columns)): 
            try:
                approximate_entropy, parameters = nk.entropy_approximate(records_AU_aft_cleaned[records_AU_aft_cleaned.columns[i]])
                 # Approximate entropy
            except:
                approximate_entropy = 0
            app_ent_AU_aft.append(approximate_entropy)
        data_approx_entropy_aft.loc[0] = app_ent_AU_aft

        data_action_aft = pd.concat([data_action_aft, data_approx_entropy_aft], axis=1)

        rsd_columns_aft= [name + '_aft' for name in records_AU_aft_cleaned.columns]
        data_rsd_aft = pd.DataFrame(columns=rsd_columns_aft)
        rsd_AU_aft = []
        for i in range(0, len(rsd_columns_aft)): 
            rsd = 100*np.std(records_AU_aft_cleaned[records_AU_aft_cleaned.columns[i]])/(np.mean(records_AU_aft_cleaned[records_AU_aft_cleaned.columns[i]])+0.00000000000000000000001)
            rsd_AU_aft.append(rsd)
        data_rsd_aft.loc[0] = rsd_AU_aft
        data_action_aft = pd.concat([data_action_aft, data_rsd_aft], axis=1)

        # Create evening AU data

        COLUMN_NAMES = ['AU01', 'AU02', 'AU04', 'AU06', 'AU07', 'AU10', 'AU12', 'AU14', 'AU15', 'AU17', 'AU23', 'AU24']
        COLUMN_NAMES_eve = []
        for i in range(0, len(COLUMN_NAMES)):
            COLUMN_NAMES_eve.append(COLUMN_NAMES[i] +'_eve')
        records_AU_eve = pd.DataFrame(columns= COLUMN_NAMES_eve)

        for i in range(0, len(data_evening)):
            au_data = data_evening[i]['au']
            au_names = list(au_data.keys())
            au_values = list(au_data.values())
            # Convert logit values (au_values) to probabilities
            au_probabilities = [1/(1 + np.exp(-v)) for v in au_values]

            if len(au_probabilities)!=0:
                records_AU_eve.loc[i] = au_probabilities
            else: 
                records_AU_eve.loc[i] = [np.nan]*12

        records_AU_eve_cleaned = records_AU_eve.copy()
        records_AU_eve_cleaned = records_AU_eve_cleaned.dropna()

        if len(records_AU_eve_cleaned) >=12:     # Retrieves a pre-defined feature configuration file to extract the temporal, statistical and spectral feature sets
            cfg = tsfel.get_features_by_domain()

            # Extract features
            X = tsfel.time_series_features_extractor(cfg, records_AU_eve_cleaned)
            X_eve_to_delete = []
            for name in X.columns:
                if 'Spectrogram mean coefficient_' in name:
                    X_eve_to_delete.append(name)
            X = X.drop(X_eve_to_delete, axis=1)

        else:
            X = pd.DataFrame(columns=tself_columns_eve)
            X.loc[0] = [np.nan]*1488

        data_action_eve = X.copy()

        if len(records_AU_eve_cleaned)>3:    
            for j in range(0, len(COLUMN_NAMES_eve)):
                name_action = COLUMN_NAMES_eve[j]
                features_pycatch = pycatch22.catch22_all(records_AU_eve_cleaned[name_action])
                COLUMN = []
                for i in range(0, len(features_pycatch['names'])):
                    COLUMN.append(name_action+ '_' + features_pycatch['names'][i]) 
                features_action_sub_eve = pd.DataFrame(columns=COLUMN)
                features_action_sub_eve.loc[0] = features_pycatch['values']
                if j == 0:
                    features_action_eve = features_action_sub_eve.copy()
                else:
                    features_action_eve = pd.concat([features_action_eve, features_action_sub_eve], axis=1)
        else:
            features_action_eve = pd.DataFrame(columns=pycatch_columns_eve)
            features_action_eve.loc[0] = [np.nan]*264

        data_action_eve = pd.concat([data_action_eve, features_action_eve], axis=1)
        approx_entropy_columns = [name + '_app_ent' for name in records_AU_eve_cleaned.columns]
        data_approx_entropy_eve = pd.DataFrame(columns=approx_entropy_columns)
        app_ent_AU_eve= []

        for i in range(0, len(approx_entropy_columns)): 
            try:
                approximate_entropy, parameters = nk.entropy_approximate(records_AU_eve_cleaned[records_AU_eve_cleaned.columns[i]])
                 # Approximate entropy
            except:
                approximate_entropy = 0
            app_ent_AU_eve.append(approximate_entropy)
        data_approx_entropy_eve.loc[0] = app_ent_AU_eve
        data_action_eve = pd.concat([data_action_eve, data_approx_entropy_eve], axis=1)

        rsd_columns_eve= [name + '_rsd' for name in records_AU_eve_cleaned.columns]
        data_rsd_eve = pd.DataFrame(columns=rsd_columns_eve)
        rsd_AU_eve = []
        for i in range(0, len(rsd_columns_eve)): 
            rsd = 100*np.std(records_AU_eve_cleaned[records_AU_eve_cleaned.columns[i]])/(np.mean(records_AU_eve_cleaned[records_AU_eve_cleaned.columns[i]])+0.00000000000000000000001)
            rsd_AU_eve.append(rsd)
        data_rsd_eve.loc[0] = rsd_AU_eve

        data_action_eve = pd.concat([data_action_eve, data_rsd_eve], axis=1)



        data_AU = pd.concat([data_action_mid, data_action_mor, data_action_aft, data_action_eve], axis=1)
        information_record = pd.DataFrame(columns = ['patient', 'record', 'diagnosis', 'timestamp_start', 'timestamp_end', 'type_data', 'subrecord'])
        information_record.loc[0] = [patient, record, diagnosis, timestamp_start, timestamp_end, 'test', sample]
        data_AU = pd.concat([information_record, data_AU], axis=1, join='inner')


        data_all_AU = data_AU.copy()

    if len(sub_data)>1:
        flag_sub = 0
    else:
        flag_sub = -1

    if flag_sub!=-1: 
        for sample in range(1, len(sub_data)):
            start_index = sub_data.loc[sample]['start_subrecord']
            end_index = sub_data.loc[sample]['end_subrecord']
            data_test = data3[start_index:end_index+1]
            # Select the day for 4 periods: midnight (12am-6am), morning (6am-12pm), afternoon (12pm-6pm), and evening (6pm12am) (to daytime!)

            midnight_time = []
            morning_time = []
            afternoon_time = []
            evening_time = []
            for i in range(0, len(data_test)):
                hour_sample = datetime.datetime.fromtimestamp(float(data_test[i]['timestamp'])/1000).hour
                if hour_sample>=0 and hour_sample<6:
                    midnight_time.append(i)
                if hour_sample>=6 and hour_sample<12:
                    morning_time.append(i)
                if hour_sample>=12 and hour_sample<18:
                    afternoon_time.append(i)
                if hour_sample>=18 and hour_sample<=23:
                    evening_time.append(i)

            data_midnight = []
            for i in range(0, len(midnight_time)):
                data_midnight.append(data_test[midnight_time[i]])

            data_morning = []
            for i in range(0, len(morning_time)):
                data_morning.append(data_test[morning_time[i]])

            data_afternoon = []
            for i in range(0, len(afternoon_time)):
                data_afternoon.append(data_test[afternoon_time[i]])

            data_evening = []
            for i in range(0, len(evening_time)):
                data_evening.append(data_test[evening_time[i]])





            # Define separete subdata for the midningt, morning, afternoon and evening 

            COLUMN_NAMES = ['AU01', 'AU02', 'AU04', 'AU06', 'AU07', 'AU10', 'AU12', 'AU14', 'AU15', 'AU17', 'AU23', 'AU24']
            COLUMN_NAMES_mid = []
            for i in range(0, len(COLUMN_NAMES)):
                COLUMN_NAMES_mid.append(COLUMN_NAMES[i] +'_mid')
            records_AU_mid = pd.DataFrame(columns= COLUMN_NAMES_mid)

            for i in range(0, len(data_midnight)):
                au_data = data_midnight[i]['au']
                au_names = list(au_data.keys())
                au_values = list(au_data.values())
                # Convert logit values (au_values) to probabilities
                au_probabilities = [1/(1 + np.exp(-v)) for v in au_values]

                if len(au_probabilities)!=0:
                    records_AU_mid.loc[i] = au_probabilities
                else: 
                    records_AU_mid.loc[i] = [np.nan]*12

            records_AU_mid_cleaned = records_AU_mid.copy()
            records_AU_mid_cleaned = records_AU_mid_cleaned.dropna()

            if len(records_AU_mid_cleaned) >=12:     # Retrieves a pre-defined feature configuration file to extract the temporal, statistical and spectral feature sets
                cfg = tsfel.get_features_by_domain()

                # Extract features
                X = tsfel.time_series_features_extractor(cfg, records_AU_mid_cleaned)
                X_mid_to_delete = []
                for name in X.columns:
                    if 'Spectrogram mean coefficient_' in name:
                        X_mid_to_delete.append(name)
                X = X.drop(X_mid_to_delete, axis=1)

            else:
                X = pd.DataFrame(columns=tself_columns_mid)
                X.loc[0] = [np.nan]*1488

            data_action_mid = X.copy()

            if len(records_AU_mid_cleaned)>3:    
                for j in range(0, len(COLUMN_NAMES_mid)):
                    name_action = COLUMN_NAMES_mid[j]
                    features_pycatch = pycatch22.catch22_all(records_AU_mid_cleaned[name_action])
                    COLUMN = []
                    for i in range(0, len(features_pycatch['names'])):
                        COLUMN.append(name_action+ '_' + features_pycatch['names'][i]) 
                    features_action_sub_mid = pd.DataFrame(columns=COLUMN)
                    features_action_sub_mid.loc[0] = features_pycatch['values']
                    if j == 0:
                        features_action_mid = features_action_sub_mid.copy()
                    else:
                        features_action_mid = pd.concat([features_action_mid, features_action_sub_mid], axis=1)
            else:
                features_action_mid = pd.DataFrame(columns=pycatch_columns_mid)
                features_action_mid.loc[0] = [np.nan]*264 


            data_action_mid = pd.concat([data_action_mid, features_action_mid], axis=1)

            approx_entropy_columns = [name + '_app_ent' for name in records_AU_mid_cleaned.columns]
            data_approx_entropy_mid = pd.DataFrame(columns=approx_entropy_columns)
            app_ent_AU_mid= []

            for i in range(0, len(approx_entropy_columns)): 
                try:
                    approximate_entropy, parameters = nk.entropy_approximate(records_AU_mid_cleaned[records_AU_mid_cleaned.columns[i]])
                     # Approximate entropy
                except:
                    approximate_entropy = 0
                app_ent_AU_mid.append(approximate_entropy)
            data_approx_entropy_mid.loc[0] = app_ent_AU_mid

            data_action_mid = pd.concat([data_action_mid, data_approx_entropy_mid], axis=1)

            rsd_columns_mid = [name + '_rsd' for name in records_AU_mid_cleaned.columns]
            data_rsd_mid = pd.DataFrame(columns=rsd_columns_mid)
            rsd_AU_mid = []
            for i in range(0, len(rsd_columns_mid)): 
                rsd = 100*np.std(records_AU_mid_cleaned[records_AU_mid_cleaned.columns[i]])/(np.mean(records_AU_mid_cleaned[records_AU_mid_cleaned.columns[i]])+0.00000000000000000000001)
                rsd_AU_mid.append(rsd)
            data_rsd_mid.loc[0] = rsd_AU_mid

            data_action_mid = pd.concat([data_action_mid, data_rsd_mid], axis=1)

            # Extract feautres for morning
            COLUMN_NAMES = ['AU01', 'AU02', 'AU04', 'AU06', 'AU07', 'AU10', 'AU12', 'AU14', 'AU15', 'AU17', 'AU23', 'AU24']
            COLUMN_NAMES_mor = []
            for i in range(0, len(COLUMN_NAMES)):
                COLUMN_NAMES_mor.append(COLUMN_NAMES[i] +'_mor')
            records_AU_mor = pd.DataFrame(columns= COLUMN_NAMES_mor)
            for i in range(0, len(data_morning)):
                au_data = data_morning[i]['au']
                au_names = list(au_data.keys())
                au_values = list(au_data.values())
                # Convert logit values (au_values) to probabilities
                au_probabilities = [1/(1 + np.exp(-v)) for v in au_values]

                if len(au_probabilities)!=0:
                    records_AU_mor.loc[i] = au_probabilities
                else: 
                    records_AU_mor.loc[i] = [np.nan]*12

            records_AU_mor_cleaned = records_AU_mor.copy()
            records_AU_mor_cleaned = records_AU_mor_cleaned.dropna()

            print(records_AU_mor_cleaned)

            if len(records_AU_mor_cleaned) >=12:     # Retrieves a pre-defined feature configuration file to extract the temporal, statistical and spectral feature sets
                cfg = tsfel.get_features_by_domain()

                # Extract features
                X = tsfel.time_series_features_extractor(cfg, records_AU_mor_cleaned)
                X_mor_to_delete = []
                for name in X.columns:
                    if 'Spectrogram mean coefficient_' in name:
                        X_mor_to_delete.append(name)
                X = X.drop(X_mor_to_delete, axis=1)

            else:
                X = pd.DataFrame(columns=tself_columns_mor)
                X.loc[0] = [np.nan]*1488

            data_action_mor = X.copy()

            if len(records_AU_mor_cleaned)>3:    
                for j in range(0, len(COLUMN_NAMES_mor)):
                    name_action = COLUMN_NAMES_mor[j]
                    features_pycatch = pycatch22.catch22_all(records_AU_mor_cleaned[name_action])
                    COLUMN = []
                    for i in range(0, len(features_pycatch['names'])):
                        COLUMN.append(name_action+ '_' + features_pycatch['names'][i]) 
                    features_action_sub_mor = pd.DataFrame(columns=COLUMN)
                    features_action_sub_mor.loc[0] = features_pycatch['values']
                    if j == 0:
                        features_action_mor = features_action_sub_mor.copy()
                    else:
                        features_action_mor = pd.concat([features_action_mor, features_action_sub_mor], axis=1)
            else:
                features_action_mor = pd.DataFrame(columns=pycatch_columns_mor)
                features_action_mor.loc[0] = [np.nan]*264




            data_action_mor = pd.concat([data_action_mor, features_action_mor], axis=1)

            approx_entropy_columns = [name + '_app_ent' for name in records_AU_mor_cleaned.columns]
            data_approx_entropy_mor = pd.DataFrame(columns=approx_entropy_columns)
            app_ent_AU_mor= []

            for i in range(0, len(approx_entropy_columns)): 
                try:
                    approximate_entropy, parameters = nk.entropy_approximate(records_AU_mor_cleaned[records_AU_mor_cleaned.columns[i]])
                     # Approximate entropy
                except:
                    approximate_entropy = 0
                app_ent_AU_mor.append(approximate_entropy)
            data_approx_entropy_mor.loc[0] = app_ent_AU_mor
            data_action_mor = pd.concat([data_action_mor, data_approx_entropy_mor], axis=1)
            rsd_columns_mor= [name + '_rsd' for name in records_AU_mor_cleaned.columns]
            data_rsd_mor = pd.DataFrame(columns=rsd_columns_mor)
            rsd_AU_mor = []
            for i in range(0, len(rsd_columns_mor)): 
                rsd = 100*np.std(records_AU_mor_cleaned[records_AU_mor_cleaned.columns[i]])/(np.mean(records_AU_mor_cleaned[records_AU_mor_cleaned.columns[i]])+0.00000000000000000000001)
                rsd_AU_mor.append(rsd)
            data_rsd_mor.loc[0] = rsd_AU_mor

            data_action_mor = pd.concat([data_action_mor, data_rsd_mor], axis=1)
            # Create records for afternoon

            COLUMN_NAMES = ['AU01', 'AU02', 'AU04', 'AU06', 'AU07', 'AU10', 'AU12', 'AU14', 'AU15', 'AU17', 'AU23', 'AU24']
            COLUMN_NAMES_aft = []
            for i in range(0, len(COLUMN_NAMES)):
                COLUMN_NAMES_aft.append(COLUMN_NAMES[i] +'_aft')
            records_AU_aft = pd.DataFrame(columns= COLUMN_NAMES_aft)

            for i in range(0, len(data_afternoon)):
                au_data = data_afternoon[i]['au']
                au_names = list(au_data.keys())
                au_values = list(au_data.values())
                # Convert logit values (au_values) to probabilities
                au_probabilities = [1/(1 + np.exp(-v)) for v in au_values]

                if len(au_probabilities)!=0:
                    records_AU_aft.loc[i] = au_probabilities
                else: 
                    records_AU_aft.loc[i] = [np.nan]*12

            records_AU_aft_cleaned = records_AU_aft.copy()
            records_AU_aft_cleaned = records_AU_aft_cleaned.dropna()

            if len(records_AU_aft_cleaned) >=12:     # Retrieves a pre-defined feature configuration file to extract the temporal, statistical and spectral feature sets
                cfg = tsfel.get_features_by_domain()

                # Extract features
                X = tsfel.time_series_features_extractor(cfg, records_AU_aft_cleaned)
                X_aft_to_delete = []
                for name in X.columns:
                    if 'Spectrogram mean coefficient_' in name:
                        X_aft_to_delete.append(name)
                X = X.drop(X_aft_to_delete, axis=1)

            else:
                X = pd.DataFrame(columns=tself_columns_aft)
                X.loc[0] = [np.nan]*1488

            data_action_aft = X.copy()

            if len(records_AU_aft_cleaned)>3:    
                for j in range(0, len(COLUMN_NAMES_aft)):
                    name_action = COLUMN_NAMES_aft[j]
                    features_pycatch = pycatch22.catch22_all(records_AU_aft_cleaned[name_action])
                    COLUMN = []
                    for i in range(0, len(features_pycatch['names'])):
                        COLUMN.append(name_action+ '_' + features_pycatch['names'][i]) 
                    features_action_sub_aft = pd.DataFrame(columns=COLUMN)
                    features_action_sub_aft.loc[0] = features_pycatch['values']
                    if j == 0:
                        features_action_aft = features_action_sub_aft.copy()
                    else:
                        features_action_aft = pd.concat([features_action_aft, features_action_sub_aft], axis=1)
            else:
                features_action_aft = pd.DataFrame(columns=pycatch_columns_aft)
                features_action_aft.loc[0] = [np.nan]*264

            data_action_aft = pd.concat([data_action_aft, features_action_aft], axis=1)
            approx_entropy_columns = [name + '_app_ent' for name in records_AU_aft_cleaned.columns]
            data_approx_entropy_aft = pd.DataFrame(columns=approx_entropy_columns)
            app_ent_AU_aft= []


            for i in range(0, len(approx_entropy_columns)): 
                try:
                    approximate_entropy, parameters = nk.entropy_approximate(records_AU_aft_cleaned[records_AU_aft_cleaned.columns[i]])
                     # Approximate entropy
                except:
                    approximate_entropy = 0
                app_ent_AU_aft.append(approximate_entropy)
            data_approx_entropy_aft.loc[0] = app_ent_AU_aft

            data_action_aft = pd.concat([data_action_aft, data_approx_entropy_aft], axis=1)

            rsd_columns_aft= [name + '_aft' for name in records_AU_aft_cleaned.columns]
            data_rsd_aft = pd.DataFrame(columns=rsd_columns_aft)
            rsd_AU_aft = []
            for i in range(0, len(rsd_columns_aft)): 
                rsd = 100*np.std(records_AU_aft_cleaned[records_AU_aft_cleaned.columns[i]])/(np.mean(records_AU_aft_cleaned[records_AU_aft_cleaned.columns[i]])+0.00000000000000000000001)
                rsd_AU_aft.append(rsd)
            data_rsd_aft.loc[0] = rsd_AU_aft
            data_action_aft = pd.concat([data_action_aft, data_rsd_aft], axis=1)

            # Create evening AU data

            COLUMN_NAMES = ['AU01', 'AU02', 'AU04', 'AU06', 'AU07', 'AU10', 'AU12', 'AU14', 'AU15', 'AU17', 'AU23', 'AU24']
            COLUMN_NAMES_eve = []
            for i in range(0, len(COLUMN_NAMES)):
                COLUMN_NAMES_eve.append(COLUMN_NAMES[i] +'_eve')
            records_AU_eve = pd.DataFrame(columns= COLUMN_NAMES_eve)

            for i in range(0, len(data_evening)):
                au_data = data_evening[i]['au']
                au_names = list(au_data.keys())
                au_values = list(au_data.values())
                # Convert logit values (au_values) to probabilities
                au_probabilities = [1/(1 + np.exp(-v)) for v in au_values]

                if len(au_probabilities)!=0:
                    records_AU_eve.loc[i] = au_probabilities
                else: 
                    records_AU_eve.loc[i] = [np.nan]*12

            records_AU_eve_cleaned = records_AU_eve.copy()
            records_AU_eve_cleaned = records_AU_eve_cleaned.dropna()

            if len(records_AU_eve_cleaned) >=12:     # Retrieves a pre-defined feature configuration file to extract the temporal, statistical and spectral feature sets
                cfg = tsfel.get_features_by_domain()

                # Extract features
                X = tsfel.time_series_features_extractor(cfg, records_AU_eve_cleaned)
                X_eve_to_delete = []
                for name in X.columns:
                    if 'Spectrogram mean coefficient_' in name:
                        X_eve_to_delete.append(name)
                X = X.drop(X_eve_to_delete, axis=1)

            else:
                X = pd.DataFrame(columns=tself_columns_eve)
                X.loc[0] = [np.nan]*1488

            data_action_eve = X.copy()

            if len(records_AU_eve_cleaned)>3:    
                for j in range(0, len(COLUMN_NAMES_eve)):
                    name_action = COLUMN_NAMES_eve[j]
                    features_pycatch = pycatch22.catch22_all(records_AU_eve_cleaned[name_action])
                    COLUMN = []
                    for i in range(0, len(features_pycatch['names'])):
                        COLUMN.append(name_action+ '_' + features_pycatch['names'][i]) 
                    features_action_sub_eve = pd.DataFrame(columns=COLUMN)
                    features_action_sub_eve.loc[0] = features_pycatch['values']
                    if j == 0:
                        features_action_eve = features_action_sub_eve.copy()
                    else:
                        features_action_eve = pd.concat([features_action_eve, features_action_sub_eve], axis=1)
            else:
                features_action_eve = pd.DataFrame(columns=pycatch_columns_eve)
                features_action_eve.loc[0] = [np.nan]*264

            data_action_eve = pd.concat([data_action_eve, features_action_eve], axis=1)
            approx_entropy_columns = [name + '_app_ent' for name in records_AU_eve_cleaned.columns]
            data_approx_entropy_eve = pd.DataFrame(columns=approx_entropy_columns)
            app_ent_AU_eve= []

            for i in range(0, len(approx_entropy_columns)): 
                try:
                    approximate_entropy, parameters = nk.entropy_approximate(records_AU_eve_cleaned[records_AU_eve_cleaned.columns[i]])
                     # Approximate entropy
                except:
                    approximate_entropy = 0
                app_ent_AU_eve.append(approximate_entropy)
            data_approx_entropy_eve.loc[0] = app_ent_AU_eve
            data_action_eve = pd.concat([data_action_eve, data_approx_entropy_eve], axis=1)

            rsd_columns_eve= [name + '_rsd' for name in records_AU_eve_cleaned.columns]
            data_rsd_eve = pd.DataFrame(columns=rsd_columns_eve)
            rsd_AU_eve = []
            for i in range(0, len(rsd_columns_eve)): 
                rsd = 100*np.std(records_AU_eve_cleaned[records_AU_eve_cleaned.columns[i]])/(np.mean(records_AU_eve_cleaned[records_AU_eve_cleaned.columns[i]])+0.00000000000000000000001)
                rsd_AU_eve.append(rsd)
            data_rsd_eve.loc[0] = rsd_AU_eve

            data_action_eve = pd.concat([data_action_eve, data_rsd_eve], axis=1)



            data_AU = pd.concat([data_action_mid, data_action_mor, data_action_aft, data_action_eve], axis=1)
            information_record = pd.DataFrame(columns = ['patient', 'record', 'diagnosis', 'timestamp_start', 'timestamp_end', 'type_data', 'subrecord'])
            information_record.loc[0] = [patient, record, diagnosis, timestamp_start, timestamp_end, 'test', sample]
            data_AU = pd.concat([information_record, data_AU], axis=1, join='inner')


            data_all_AU = pd.concat([data_all_AU, data_AU])

            start_index = 0
            end_index = sub_data.loc[sample]['start_subrecord']
            data_train = data3[start_index:end_index+1]
            # Select the day for 4 periods: midnight (12am-6am), morning (6am-12pm), afternoon (12pm-6pm), and evening (6pm12am) (to daytime!)

            midnight_time = []
            morning_time = []
            afternoon_time = []
            evening_time = []
            for i in range(0, len(data_train)):
                hour_sample = datetime.datetime.fromtimestamp(float(data_train[i]['timestamp'])/1000).hour
                if hour_sample>=0 and hour_sample<6:
                    midnight_time.append(i)
                if hour_sample>=6 and hour_sample<12:
                    morning_time.append(i)
                if hour_sample>=12 and hour_sample<18:
                    afternoon_time.append(i)
                if hour_sample>=18 and hour_sample<=23:
                    evening_time.append(i)

            data_midnight = []
            for i in range(0, len(midnight_time)):
                data_midnight.append(data_train[midnight_time[i]])

            data_morning = []
            for i in range(0, len(morning_time)):
                data_morning.append(data_train[morning_time[i]])

            data_afternoon = []
            for i in range(0, len(afternoon_time)):
                data_afternoon.append(data_train[afternoon_time[i]])

            data_evening = []
            for i in range(0, len(evening_time)):
                data_evening.append(data_train[evening_time[i]])





            # Define separete subdata for the midningt, morning, afternoon and evening 

            COLUMN_NAMES = ['AU01', 'AU02', 'AU04', 'AU06', 'AU07', 'AU10', 'AU12', 'AU14', 'AU15', 'AU17', 'AU23', 'AU24']
            COLUMN_NAMES_mid = []
            for i in range(0, len(COLUMN_NAMES)):
                COLUMN_NAMES_mid.append(COLUMN_NAMES[i] +'_mid')
            records_AU_mid = pd.DataFrame(columns= COLUMN_NAMES_mid)

            for i in range(0, len(data_midnight)):
                au_data = data_midnight[i]['au']
                au_names = list(au_data.keys())
                au_values = list(au_data.values())
                # Convert logit values (au_values) to probabilities
                au_probabilities = [1/(1 + np.exp(-v)) for v in au_values]

                if len(au_probabilities)!=0:
                    records_AU_mid.loc[i] = au_probabilities
                else: 
                    records_AU_mid.loc[i] = [np.nan]*12

            records_AU_mid_cleaned = records_AU_mid.copy()
            records_AU_mid_cleaned = records_AU_mid_cleaned.dropna()

            if len(records_AU_mid_cleaned) >=12:     # Retrieves a pre-defined feature configuration file to extract the temporal, statistical and spectral feature sets
                cfg = tsfel.get_features_by_domain()

                # Extract features
                X = tsfel.time_series_features_extractor(cfg, records_AU_mid_cleaned)
                X_mid_to_delete = []
                for name in X.columns:
                    if 'Spectrogram mean coefficient_' in name:
                        X_mid_to_delete.append(name)
                X = X.drop(X_mid_to_delete, axis=1)

            else:
                X = pd.DataFrame(columns=tself_columns_mid)
                X.loc[0] = [np.nan]*1488

            data_action_mid = X.copy()

            if len(records_AU_mid_cleaned)>3:    
                for j in range(0, len(COLUMN_NAMES_mid)):
                    name_action = COLUMN_NAMES_mid[j]
                    features_pycatch = pycatch22.catch22_all(records_AU_mid_cleaned[name_action])
                    COLUMN = []
                    for i in range(0, len(features_pycatch['names'])):
                        COLUMN.append(name_action+ '_' + features_pycatch['names'][i]) 
                    features_action_sub_mid = pd.DataFrame(columns=COLUMN)
                    features_action_sub_mid.loc[0] = features_pycatch['values']
                    if j == 0:
                        features_action_mid = features_action_sub_mid.copy()
                    else:
                        features_action_mid = pd.concat([features_action_mid, features_action_sub_mid], axis=1)
            else:
                features_action_mid = pd.DataFrame(columns=pycatch_columns_mid)
                features_action_mid.loc[0] = [np.nan]*264 


            data_action_mid = pd.concat([data_action_mid, features_action_mid], axis=1)

            approx_entropy_columns = [name + '_app_ent' for name in records_AU_mid_cleaned.columns]
            data_approx_entropy_mid = pd.DataFrame(columns=approx_entropy_columns)
            app_ent_AU_mid= []

            for i in range(0, len(approx_entropy_columns)): 
                try:
                    approximate_entropy, parameters = nk.entropy_approximate(records_AU_mid_cleaned[records_AU_mid_cleaned.columns[i]])
                     # Approximate entropy
                except:
                    approximate_entropy = 0
                app_ent_AU_mid.append(approximate_entropy)
            data_approx_entropy_mid.loc[0] = app_ent_AU_mid

            data_action_mid = pd.concat([data_action_mid, data_approx_entropy_mid], axis=1)

            rsd_columns_mid = [name + '_rsd' for name in records_AU_mid_cleaned.columns]
            data_rsd_mid = pd.DataFrame(columns=rsd_columns_mid)
            rsd_AU_mid = []
            for i in range(0, len(rsd_columns_mid)): 
                rsd = 100*np.std(records_AU_mid_cleaned[records_AU_mid_cleaned.columns[i]])/(np.mean(records_AU_mid_cleaned[records_AU_mid_cleaned.columns[i]])+0.00000000000000000000001)
                rsd_AU_mid.append(rsd)
            data_rsd_mid.loc[0] = rsd_AU_mid

            data_action_mid = pd.concat([data_action_mid, data_rsd_mid], axis=1)

            # Extract feautres for morning
            COLUMN_NAMES = ['AU01', 'AU02', 'AU04', 'AU06', 'AU07', 'AU10', 'AU12', 'AU14', 'AU15', 'AU17', 'AU23', 'AU24']
            COLUMN_NAMES_mor = []
            for i in range(0, len(COLUMN_NAMES)):
                COLUMN_NAMES_mor.append(COLUMN_NAMES[i] +'_mor')
            records_AU_mor = pd.DataFrame(columns= COLUMN_NAMES_mor)
            for i in range(0, len(data_morning)):
                au_data = data_morning[i]['au']
                au_names = list(au_data.keys())
                au_values = list(au_data.values())
                # Convert logit values (au_values) to probabilities
                au_probabilities = [1/(1 + np.exp(-v)) for v in au_values]

                if len(au_probabilities)!=0:
                    records_AU_mor.loc[i] = au_probabilities
                else: 
                    records_AU_mor.loc[i] = [np.nan]*12

            records_AU_mor_cleaned = records_AU_mor.copy()
            records_AU_mor_cleaned = records_AU_mor_cleaned.dropna()

            print(records_AU_mor_cleaned)

            if len(records_AU_mor_cleaned) >=12:     # Retrieves a pre-defined feature configuration file to extract the temporal, statistical and spectral feature sets
                cfg = tsfel.get_features_by_domain()

                # Extract features
                X = tsfel.time_series_features_extractor(cfg, records_AU_mor_cleaned)
                X_mor_to_delete = []
                for name in X.columns:
                    if 'Spectrogram mean coefficient_' in name:
                        X_mor_to_delete.append(name)
                X = X.drop(X_mor_to_delete, axis=1)

            else:
                X = pd.DataFrame(columns=tself_columns_mor)
                X.loc[0] = [np.nan]*1488

            data_action_mor = X.copy()

            if len(records_AU_mor_cleaned)>3:    
                for j in range(0, len(COLUMN_NAMES_mor)):
                    name_action = COLUMN_NAMES_mor[j]
                    features_pycatch = pycatch22.catch22_all(records_AU_mor_cleaned[name_action])
                    COLUMN = []
                    for i in range(0, len(features_pycatch['names'])):
                        COLUMN.append(name_action+ '_' + features_pycatch['names'][i]) 
                    features_action_sub_mor = pd.DataFrame(columns=COLUMN)
                    features_action_sub_mor.loc[0] = features_pycatch['values']
                    if j == 0:
                        features_action_mor = features_action_sub_mor.copy()
                    else:
                        features_action_mor = pd.concat([features_action_mor, features_action_sub_mor], axis=1)
            else:
                features_action_mor = pd.DataFrame(columns=pycatch_columns_mor)
                features_action_mor.loc[0] = [np.nan]*264




            data_action_mor = pd.concat([data_action_mor, features_action_mor], axis=1)

            approx_entropy_columns = [name + '_app_ent' for name in records_AU_mor_cleaned.columns]
            data_approx_entropy_mor = pd.DataFrame(columns=approx_entropy_columns)
            app_ent_AU_mor= []

            for i in range(0, len(approx_entropy_columns)): 
                try:
                    approximate_entropy, parameters = nk.entropy_approximate(records_AU_mor_cleaned[records_AU_mor_cleaned.columns[i]])
                     # Approximate entropy
                except:
                    approximate_entropy = 0
                app_ent_AU_mor.append(approximate_entropy)
            data_approx_entropy_mor.loc[0] = app_ent_AU_mor
            data_action_mor = pd.concat([data_action_mor, data_approx_entropy_mor], axis=1)
            rsd_columns_mor= [name + '_rsd' for name in records_AU_mor_cleaned.columns]
            data_rsd_mor = pd.DataFrame(columns=rsd_columns_mor)
            rsd_AU_mor = []
            for i in range(0, len(rsd_columns_mor)): 
                rsd = 100*np.std(records_AU_mor_cleaned[records_AU_mor_cleaned.columns[i]])/(np.mean(records_AU_mor_cleaned[records_AU_mor_cleaned.columns[i]])+0.00000000000000000000001)
                rsd_AU_mor.append(rsd)
            data_rsd_mor.loc[0] = rsd_AU_mor

            data_action_mor = pd.concat([data_action_mor, data_rsd_mor], axis=1)
            # Create records for afternoon

            COLUMN_NAMES = ['AU01', 'AU02', 'AU04', 'AU06', 'AU07', 'AU10', 'AU12', 'AU14', 'AU15', 'AU17', 'AU23', 'AU24']
            COLUMN_NAMES_aft = []
            for i in range(0, len(COLUMN_NAMES)):
                COLUMN_NAMES_aft.append(COLUMN_NAMES[i] +'_aft')
            records_AU_aft = pd.DataFrame(columns= COLUMN_NAMES_aft)

            for i in range(0, len(data_afternoon)):
                au_data = data_afternoon[i]['au']
                au_names = list(au_data.keys())
                au_values = list(au_data.values())
                # Convert logit values (au_values) to probabilities
                au_probabilities = [1/(1 + np.exp(-v)) for v in au_values]

                if len(au_probabilities)!=0:
                    records_AU_aft.loc[i] = au_probabilities
                else: 
                    records_AU_aft.loc[i] = [np.nan]*12

            records_AU_aft_cleaned = records_AU_aft.copy()
            records_AU_aft_cleaned = records_AU_aft_cleaned.dropna()

            if len(records_AU_aft_cleaned) >=12:     # Retrieves a pre-defined feature configuration file to extract the temporal, statistical and spectral feature sets
                cfg = tsfel.get_features_by_domain()

                # Extract features
                X = tsfel.time_series_features_extractor(cfg, records_AU_aft_cleaned)
                X_aft_to_delete = []
                for name in X.columns:
                    if 'Spectrogram mean coefficient_' in name:
                        X_aft_to_delete.append(name)
                X = X.drop(X_aft_to_delete, axis=1)

            else:
                X = pd.DataFrame(columns=tself_columns_aft)
                X.loc[0] = [np.nan]*1488

            data_action_aft = X.copy()

            if len(records_AU_aft_cleaned)>3:    
                for j in range(0, len(COLUMN_NAMES_aft)):
                    name_action = COLUMN_NAMES_aft[j]
                    features_pycatch = pycatch22.catch22_all(records_AU_aft_cleaned[name_action])
                    COLUMN = []
                    for i in range(0, len(features_pycatch['names'])):
                        COLUMN.append(name_action+ '_' + features_pycatch['names'][i]) 
                    features_action_sub_aft = pd.DataFrame(columns=COLUMN)
                    features_action_sub_aft.loc[0] = features_pycatch['values']
                    if j == 0:
                        features_action_aft = features_action_sub_aft.copy()
                    else:
                        features_action_aft = pd.concat([features_action_aft, features_action_sub_aft], axis=1)
            else:
                features_action_aft = pd.DataFrame(columns=pycatch_columns_aft)
                features_action_aft.loc[0] = [np.nan]*264

            data_action_aft = pd.concat([data_action_aft, features_action_aft], axis=1)
            approx_entropy_columns = [name + '_app_ent' for name in records_AU_aft_cleaned.columns]
            data_approx_entropy_aft = pd.DataFrame(columns=approx_entropy_columns)
            app_ent_AU_aft= []


            for i in range(0, len(approx_entropy_columns)): 
                try:
                    approximate_entropy, parameters = nk.entropy_approximate(records_AU_aft_cleaned[records_AU_aft_cleaned.columns[i]])
                     # Approximate entropy
                except:
                    approximate_entropy = 0
                app_ent_AU_aft.append(approximate_entropy)
            data_approx_entropy_aft.loc[0] = app_ent_AU_aft

            data_action_aft = pd.concat([data_action_aft, data_approx_entropy_aft], axis=1)

            rsd_columns_aft= [name + '_aft' for name in records_AU_aft_cleaned.columns]
            data_rsd_aft = pd.DataFrame(columns=rsd_columns_aft)
            rsd_AU_aft = []
            for i in range(0, len(rsd_columns_aft)): 
                rsd = 100*np.std(records_AU_aft_cleaned[records_AU_aft_cleaned.columns[i]])/(np.mean(records_AU_aft_cleaned[records_AU_aft_cleaned.columns[i]])+0.00000000000000000000001)
                rsd_AU_aft.append(rsd)
            data_rsd_aft.loc[0] = rsd_AU_aft
            data_action_aft = pd.concat([data_action_aft, data_rsd_aft], axis=1)

            # Create evening AU data

            COLUMN_NAMES = ['AU01', 'AU02', 'AU04', 'AU06', 'AU07', 'AU10', 'AU12', 'AU14', 'AU15', 'AU17', 'AU23', 'AU24']
            COLUMN_NAMES_eve = []
            for i in range(0, len(COLUMN_NAMES)):
                COLUMN_NAMES_eve.append(COLUMN_NAMES[i] +'_eve')
            records_AU_eve = pd.DataFrame(columns= COLUMN_NAMES_eve)

            for i in range(0, len(data_evening)):
                au_data = data_evening[i]['au']
                au_names = list(au_data.keys())
                au_values = list(au_data.values())
                # Convert logit values (au_values) to probabilities
                au_probabilities = [1/(1 + np.exp(-v)) for v in au_values]

                if len(au_probabilities)!=0:
                    records_AU_eve.loc[i] = au_probabilities
                else: 
                    records_AU_eve.loc[i] = [np.nan]*12

            records_AU_eve_cleaned = records_AU_eve.copy()
            records_AU_eve_cleaned = records_AU_eve_cleaned.dropna()

            if len(records_AU_eve_cleaned) >=12:     # Retrieves a pre-defined feature configuration file to extract the temporal, statistical and spectral feature sets
                cfg = tsfel.get_features_by_domain()

                # Extract features
                X = tsfel.time_series_features_extractor(cfg, records_AU_eve_cleaned)
                X_eve_to_delete = []
                for name in X.columns:
                    if 'Spectrogram mean coefficient_' in name:
                        X_eve_to_delete.append(name)
                X = X.drop(X_eve_to_delete, axis=1)

            else:
                X = pd.DataFrame(columns=tself_columns_eve)
                X.loc[0] = [np.nan]*1488

            data_action_eve = X.copy()

            if len(records_AU_eve_cleaned)>3:    
                for j in range(0, len(COLUMN_NAMES_eve)):
                    name_action = COLUMN_NAMES_eve[j]
                    features_pycatch = pycatch22.catch22_all(records_AU_eve_cleaned[name_action])
                    COLUMN = []
                    for i in range(0, len(features_pycatch['names'])):
                        COLUMN.append(name_action+ '_' + features_pycatch['names'][i]) 
                    features_action_sub_eve = pd.DataFrame(columns=COLUMN)
                    features_action_sub_eve.loc[0] = features_pycatch['values']
                    if j == 0:
                        features_action_eve = features_action_sub_eve.copy()
                    else:
                        features_action_eve = pd.concat([features_action_eve, features_action_sub_eve], axis=1)
            else:
                features_action_eve = pd.DataFrame(columns=pycatch_columns_eve)
                features_action_eve.loc[0] = [np.nan]*264

            data_action_eve = pd.concat([data_action_eve, features_action_eve], axis=1)
            approx_entropy_columns = [name + '_app_ent' for name in records_AU_eve_cleaned.columns]
            data_approx_entropy_eve = pd.DataFrame(columns=approx_entropy_columns)
            app_ent_AU_eve= []

            for i in range(0, len(approx_entropy_columns)): 
                try:
                    approximate_entropy, parameters = nk.entropy_approximate(records_AU_eve_cleaned[records_AU_eve_cleaned.columns[i]])
                     # Approximate entropy
                except:
                    approximate_entropy = 0
                app_ent_AU_eve.append(approximate_entropy)
            data_approx_entropy_eve.loc[0] = app_ent_AU_eve
            data_action_eve = pd.concat([data_action_eve, data_approx_entropy_eve], axis=1)

            rsd_columns_eve= [name + '_rsd' for name in records_AU_eve_cleaned.columns]
            data_rsd_eve = pd.DataFrame(columns=rsd_columns_eve)
            rsd_AU_eve = []
            for i in range(0, len(rsd_columns_eve)): 
                rsd = 100*np.std(records_AU_eve_cleaned[records_AU_eve_cleaned.columns[i]])/(np.mean(records_AU_eve_cleaned[records_AU_eve_cleaned.columns[i]])+0.00000000000000000000001)
                rsd_AU_eve.append(rsd)
            data_rsd_eve.loc[0] = rsd_AU_eve

            data_action_eve = pd.concat([data_action_eve, data_rsd_eve], axis=1)



            data_AU = pd.concat([data_action_mid, data_action_mor, data_action_aft, data_action_eve], axis=1)
            information_record = pd.DataFrame(columns = ['patient', 'record', 'diagnosis', 'timestamp_start', 'timestamp_end', 'type_data', 'subrecord'])
            information_record.loc[0] = [patient, record, diagnosis, timestamp_start, timestamp_end, 'train', sample]
            data_AU = pd.concat([information_record, data_AU], axis=1, join='inner')




            data_all_AU= pd.concat([data_all_AU, data_AU])

    data_all_AU.to_csv('dataset/AU_cross/AU_'+str(record)+'_.csv')

      AU01_mor  AU02_mor  AU04_mor  AU06_mor  AU07_mor  AU10_mor  AU12_mor  \
0     0.000015  0.000001  0.498614  0.999837  1.000000  0.000490  0.034335   
1     0.000315  0.000003  0.855699  0.999131  0.999999  0.407927  0.057168   
2     0.003255  0.000055  0.999479  0.999978  1.000000  0.012462  0.003082   
3     0.010504  0.000802  0.996087  0.996581  1.000000  0.008025  0.000259   
4     0.001721  0.000088  0.943314  0.995476  0.999996  0.413842  0.041650   
...        ...       ...       ...       ...       ...       ...       ...   
1838  0.000203  0.000516  0.015226  0.915813  0.999754  0.000096  0.807961   
1839  0.009546  0.005558  0.027269  0.050700  0.977495  0.000289  0.114816   
1840  0.005673  0.002161  0.041308  0.259924  0.999871  0.000905  0.008261   
1841  0.052564  0.000912  0.099966  0.997928  0.999995  0.170305  0.103539   
1842  0.006541  0.143057  0.000011  0.808364  0.999054  0.000187  0.056115   

      AU14_mor  AU15_mor  AU17_mor  AU23_mor  AU24_mor  
0     

C:\Users\skiju\AppData\Local\Temp\ipykernel_21688\3400928971.py:286: UserWarning: Using default sampling frequency set in configuration file.
  X = tsfel.time_series_features_extractor(cfg, records_AU_mor_cleaned)


          AU01_mor  AU02_mor      AU04_mor  AU06_mor  AU07_mor  AU10_mor  \
11    1.200772e-01  0.365272  8.637230e-06  0.106958  0.473407  0.289550   
12    9.498403e-07  0.000070  1.952511e-07  0.007870  0.199778  0.187536   
13    1.946259e-01  0.257802  7.086790e-07  0.025324  0.883951  0.066459   
14    1.656627e-02  0.022350  1.077687e-06  0.001400  0.252224  0.000077   
16    2.266855e-01  0.678923  3.262601e-02  0.154560  0.350381  0.000012   
...            ...       ...           ...       ...       ...       ...   
1198  3.011212e-03  0.002439  7.430121e-06  0.415708  0.999985  0.000002   
1200  5.826930e-03  0.003475  2.928102e-04  0.000628  0.566820  0.000005   
1203  1.844977e-04  0.010815  2.726448e-06  0.965789  0.999993  0.012147   
1204  1.654956e-07  0.000010  2.281662e-06  0.050089  0.965134  0.002500   
1205  1.032829e-06  0.000005  5.118121e-05  0.000144  0.006857  0.003512   

      AU12_mor  AU14_mor  AU15_mor  AU17_mor  AU23_mor  AU24_mor  
11    0.976204  0.00

C:\Users\skiju\AppData\Local\Temp\ipykernel_21688\3400928971.py:677: UserWarning: Using default sampling frequency set in configuration file.
  X = tsfel.time_series_features_extractor(cfg, records_AU_mor_cleaned)


C:\Users\skiju\AppData\Local\Temp\ipykernel_21688\3400928971.py:761: UserWarning: Using default sampling frequency set in configuration file.
  X = tsfel.time_series_features_extractor(cfg, records_AU_aft_cleaned)


...

C:\Users\skiju\AppData\Local\Temp\ipykernel_21688\3400928971.py:976: UserWarning: Using default sampling frequency set in configuration file.
  X = tsfel.time_series_features_extractor(cfg, records_AU_mid_cleaned)


     AU01_mor  AU02_mor  AU04_mor  AU06_mor  AU07_mor      AU10_mor  \
0    0.000869  0.001575  0.783037  0.000060  0.013743  1.482571e-07   
1    0.038852  0.067074  0.599964  0.000005  0.001906  3.545943e-05   
2    0.001347  0.001732  0.618487  0.000006  0.000134  5.969813e-08   
3    0.006351  0.017150  0.285802  0.000048  0.005937  1.149054e-06   
4    0.016742  0.048881  0.028240  0.000024  0.000028  1.173856e-06   
..        ...       ...       ...       ...       ...           ...   
171  0.001596  0.003526  0.591580  0.007844  0.410310  9.204269e-06   
172  0.002690  0.005667  0.903442  0.000036  0.002354  4.205517e-07   
173  0.007307  0.002516  0.760203  0.001611  0.467176  7.839537e-06   
174  0.004914  0.001796  0.651668  0.001371  0.408272  7.730703e-06   
175  0.002686  0.008637  0.672463  0.001041  0.007698  5.172638e-05   

         AU12_mor  AU14_mor  AU15_mor  AU17_mor  AU23_mor  AU24_mor  
0    6.769706e-08  0.000007  0.018287  0.022591  0.000751  0.000016  
1    1.

C:\Users\skiju\AppData\Local\Temp\ipykernel_21688\3400928971.py:1061: UserWarning: Using default sampling frequency set in configuration file.
  X = tsfel.time_series_features_extractor(cfg, records_AU_mor_cleaned)


C:\Users\skiju\AppData\Local\Temp\ipykernel_21688\3400928971.py:1145: UserWarning: Using default sampling frequency set in configuration file.
  X = tsfel.time_series_features_extractor(cfg, records_AU_aft_cleaned)


C:\Users\skiju\AppData\Local\Temp\ipykernel_21688\3400928971.py:1228: UserWarning: Using default sampling frequency set in configuration file.
  X = tsfel.time_series_features_extractor(cfg, records_AU_eve_cleaned)
